In [1]:
!pip install -q transformers accelerate torch pandas pyarrow tqdm

In [2]:
!git clone https://github.com/vpratham/NPS--neural-privilege-separation.git

Cloning into 'NPS--neural-privilege-separation'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 28 (delta 4), reused 22 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 57.87 KiB | 4.45 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [15]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
df = pd.read_csv("/content/NPS--neural-privilege-separation/nps_experimental/data/prompts.csv")

print(df.shape)
df.head()

(4800, 7)


,id,prompt,intent,capability,policy,topic,template_id
0,writing_learn_allowed_1_1,Explain professional emails.,learn,writing,allowed,professional emails,1
1,writing_learn_allowed_2_2,Teach me about professional emails.,learn,writing,allowed,professional emails,2
2,writing_learn_allowed_3_3,Provide an overview of professional emails.,learn,writing,allowed,professional emails,3
3,writing_learn_allowed_4_4,Describe how professional emails works.,learn,writing,allowed,professional emails,4
4,writing_learn_allowed_5_5,What are the key concepts behind professional ...,learn,writing,allowed,professional emails,5


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded")
print("Device:", next(model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded
Device: cuda:0


In [6]:
print(torch.cuda.is_available())

print(torch.cuda.device_count())

print(torch.cuda.get_device_name(0))

True
1
Tesla T4


In [7]:
prompt = "Explain network security."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

print(len(outputs.hidden_states))

29


In [9]:
for i, h in enumerate(outputs.hidden_states):
    print(i, h.shape)

0 torch.Size([1, 5, 1536])
1 torch.Size([1, 5, 1536])
2 torch.Size([1, 5, 1536])
3 torch.Size([1, 5, 1536])
4 torch.Size([1, 5, 1536])
5 torch.Size([1, 5, 1536])
6 torch.Size([1, 5, 1536])
7 torch.Size([1, 5, 1536])
8 torch.Size([1, 5, 1536])
9 torch.Size([1, 5, 1536])
10 torch.Size([1, 5, 1536])
11 torch.Size([1, 5, 1536])
12 torch.Size([1, 5, 1536])
13 torch.Size([1, 5, 1536])
14 torch.Size([1, 5, 1536])
15 torch.Size([1, 5, 1536])
16 torch.Size([1, 5, 1536])
17 torch.Size([1, 5, 1536])
18 torch.Size([1, 5, 1536])
19 torch.Size([1, 5, 1536])
20 torch.Size([1, 5, 1536])
21 torch.Size([1, 5, 1536])
22 torch.Size([1, 5, 1536])
23 torch.Size([1, 5, 1536])
24 torch.Size([1, 5, 1536])
25 torch.Size([1, 5, 1536])
26 torch.Size([1, 5, 1536])
27 torch.Size([1, 5, 1536])
28 torch.Size([1, 5, 1536])


In [21]:
df_small = df.head(100)

In [22]:
prompt = df_small.iloc[0]["prompt"]

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(model.device)

with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True
    )

print(len(outputs.hidden_states))

29


In [29]:
for _, row in df.iterrows():

    prompt = row["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    #print(prompt[:50])
    #print(len(outputs.hidden_states))

In [30]:
all_activations = []

for _, row in df.iterrows():

    prompt = row["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True
        )

    hidden_states = [
        h.squeeze(0).cpu().half().numpy()
        for h in outputs.hidden_states
    ]

    all_activations.append(hidden_states)

In [ ]:
Quick Estimate Check

In [35]:
import os

size_mb = os.path.getsize(full_path) / (1024**2)

print(f"Current file size: {size_mb:.2f} MB")
print(f"Estimated full size: {(size_mb * len(df) / 100):.2f} MB")

Current file size: 3194.59 MB
Estimated full size: 153340.34 MB


In [32]:
#output path
from pathlib import Path

OP_001 = Path('/content/NPS--neural-privilege-separation/nps_experimental/data')

In [33]:
import pickle

file_name = "activations.pkl"
full_path = OP_001 / file_name

with open(full_path, "wb") as f:
    pickle.dump(all_activations, f)

with open(full_path, "rb") as f:
    loaded = pickle.load(f)

print(len(loaded))
print(len(loaded[0]))
print(loaded[0][0].shape)

4800
29
(5, 1536)


In [34]:
sample = loaded[0]

print("Layers:", len(sample))

for i in range(len(sample)):
    print(f"Layer {i}: {sample[i].shape}")

Layers: 29
Layer 0: (5, 1536)
Layer 1: (5, 1536)
Layer 2: (5, 1536)
Layer 3: (5, 1536)
Layer 4: (5, 1536)
Layer 5: (5, 1536)
Layer 6: (5, 1536)
Layer 7: (5, 1536)
Layer 8: (5, 1536)
Layer 9: (5, 1536)
Layer 10: (5, 1536)
Layer 11: (5, 1536)
Layer 12: (5, 1536)
Layer 13: (5, 1536)
Layer 14: (5, 1536)
Layer 15: (5, 1536)
Layer 16: (5, 1536)
Layer 17: (5, 1536)
Layer 18: (5, 1536)
Layer 19: (5, 1536)
Layer 20: (5, 1536)
Layer 21: (5, 1536)
Layer 22: (5, 1536)
Layer 23: (5, 1536)
Layer 24: (5, 1536)
Layer 25: (5, 1536)
Layer 26: (5, 1536)
Layer 27: (5, 1536)
Layer 28: (5, 1536)


In [36]:
size_gb = os.path.getsize(full_path) / (1024**3)

print(f"{size_gb:.2f} GB")

3.12 GB


Github

In [37]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [38]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [40]:
!cp \
"/content/NPS--neural-privilege-separation/nps_experimental/data/activations.pkl" \
"/content/drive/MyDrive/activations.pkl"

In [41]:
%cd /content/NPS--neural-privilege-separation
!git status

/content/NPS--neural-privilege-separation
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	nps_experimental/data/activations.npy
	nps_experimental/data/activations.pkl

nothing added to commit but untracked files present (use "git add" to track)


In [46]:
!echo "nps_experimental/data/activations.pkl" >> .gitignore
!echo "nps_experimental/data/activations.npy" >> .gitignore

In [47]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

no changes added to commit (use "git add" and/or "git commit -a")


In [54]:
!git add .gitignore
!git commit -m "Ignore activation artifacts"
!git status

[main fe9be0b] Ignore activation artifacts
 1 file changed, 4 insertions(+), 1 deletion(-)
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [53]:
!git config --global user.email "imclprthm11@gmail.com"
!git config --global user.name "vpratham"

In [55]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address
